# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primary task: ranking / scoring in a hybrid ML pipeline.**

The system must answer: **which content pages should get attention first, why, and what action should be considered?** That makes ranking/scoring the final task.

The ranking is built in stages:

1. **Signal analysis** creates a leakage-safe feature set from observed search, traffic, engagement, freshness, and content data.
2. **Clustering** groups similar pages into performance archetypes and gives each page a fair peer group.
3. **Peer-relative analysis** measures how unusual a page is compared with similar pages, for example whether its CTR or engagement is weak for its archetype.
4. **Classification / prediction** estimates a future observed performance state using only information available before the decision point.
5. **Impact estimation** combines predicted risk, expected size of the change, and page exposure/value.
6. **Ranking / scoring** orders pages by expected adverse impact.
7. **Action generation** uses the archetype, predicted direction, and strongest abnormal signals to produce reason codes and a suggested action.

In short:

**observed signals → archetype → relative abnormalities → future prediction → expected impact → ranked priority → suggested action**

The suggested action is an evidence-based **intervention hypothesis**, not a proven causal effect. We can test prediction quality retrospectively and later test whether the recommended action causes improvement with a controlled or otherwise valid causal study.

In [1]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

methodology = [
    "signal analysis",
    "clustering / peer context",
    "archetype-relative deviations",
    "future-state prediction",
    "impact estimation",
    "ranking / scoring",
    "action hypothesis",
]
print("Methodology:", " -> ".join(methodology))


Rows: 30,000
Columns: 44
Unique content items: 30,000
Pseudonymized clients: 32
Methodology: signal analysis -> clustering / peer context -> archetype-relative deviations -> future-state prediction -> impact estimation -> ranking / scoring -> action hypothesis


## 2. Target or proxy

Different stages use different learning objects.

### Clustering

Clustering has **no target column**. It creates performance archetypes and peer baselines from the training data.

### Supervised prediction

The main supervised target should be an **observed future outcome**, not a manually created priority score or action label.

The intended structure is:

**past feature window → decision point → non-overlapping future outcome window**

The later warehouse target will be a future decline outcome such as `future_decline_30d`, built from observed search performance after the decision point. A continuous future-change value will also keep the size of the movement, so a small decline and a severe decline are not treated as equal.

The exact decline threshold, persistence rule, and minimum-volume floor are **not fixed in this notebook**. They will be defined in the data-contract stage and tested for sensitivity before model training.

### Starter-data proxy

The 30,000-row starter CSV is only a trailing-90-day snapshot, so it cannot provide a true future target. For Assignment 3, the transparent proxy is:

[
\text{decline\_proxy}=1
\quad \text{if} \quad
\texttt{trend\_direction = "down"}
]

This is a **defined current-window proxy**, not a future ground-truth outcome and not proof that a page needs intervention.

Because `trend_direction` comes from `trend_pct`, neither field can be used as a predictive feature when this proxy is the target.

### Ranking and action

The final ranking will not learn a hand-written priority label. Conceptually:

[
\text{priority}
\propto
P(\text{future decline})
\times
E(\text{decline magnitude})
\times
\text{measured exposure/value}
]

The exact scaling will be fixed later and validated rather than assigned arbitrary weights.

The action stage maps the page's archetype, predicted future state, and strongest peer-relative abnormalities to a **recommended intervention hypothesis** that can later be tested.

In [2]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_change_magnitude",
    ],
    "role": [
        "future-state classification target",
        "future-movement magnitude outcome",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


Starter proxy: decline_proxy = 1 when trend_direction == 'down'.
This is a current-window proxy, not a future causal or intervention label.

decline_proxy
0    13738
1    16262
Name: count, dtype: int64
Proxy positive rate: 54.2%



,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,decline_proxy
0,content_304f48230142,987,578,-41.4,down,1
1,content_a1fb4e703a9e,5915,2501,-57.7,down,1
2,content_9aa793d4d895,6089,2382,-60.9,down,1
3,content_331d6c4de07b,4206,3626,-13.8,stable,0
4,content_d99b7a2d90ca,6452,4211,-34.7,down,1
5,content_d4084a4bc775,1009,617,-38.9,down,1
6,content_9a34b442b552,13,1,-92.3,down,1
7,content_a63219c6e95a,632,636,0.6,stable,0
8,content_5e6c160719bc,13828,5696,-58.8,down,1
9,content_c27558df2b0c,356,252,-29.2,down,1


Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']


,field,role,available_in_starter_snapshot
0,future_decline_30d,future-state classification target,false
1,future_change_magnitude,future-movement magnitude outcome,false


## 3. Success metric

### Metric hierarchy

There is **one primary project metric** and **one primary metric for each sub-task**. Secondary metrics are diagnostics only: they help explain failure, instability, calibration, coverage or data problems, but they do not replace the primary success criterion after results are seen.

The final decision is: **which pages should limited human review capacity inspect first?** Therefore the project-level metric is:

$$\boxed{\text{Precision@50}}$$

where Precision@50 is the fraction of the 50 highest-ranked pages that later show the frozen observed adverse outcome.

The learned system is useful only if, on the same honest held-out future rows,

$$\boxed{\Delta P@50 = P@50_{\text{learned ranking}} - P@50_{\text{frozen rule baseline}} > 0}$$

`K = 50` is the pre-declared Assignment-3 reporting depth. It is not claimed to be FlyRank's true operational review capacity. If a real capacity is supplied later independently of test performance, that value can replace 50 prospectively.

### One primary metric per task / sub-task

| Task / sub-task | **Primary metric — only one** | Exact comparison | What counts as success | Why this is the primary metric |
|---|---|---|---|---|
| **Signal analysis / feature-set construction** | **Delta Precision@50 from the validated signal set** | Same downstream learner with the validated signal set versus the same learner using only the pre-declared minimal/raw feature set | `Delta P@50_signals > 0` | Signal analysis exists to improve the final review queue. This tests downstream usefulness directly instead of declaring a signal useful because of correlation alone. |
| **Clustering / archetypes** | **Delta Precision@50 from cluster-derived context** | Same downstream pipeline with cluster/archetype features versus the identical pipeline with those features removed | `Delta P@50_cluster > 0` | Silhouette can reward geometrically neat but operationally useless clusters. The cluster stage earns its place only if its peer context improves the queue. |
| **Peer-relative abnormality** | **Delta Precision@50 from peer-relative features** | Same downstream pipeline with peer-relative deviations versus the identical pipeline using raw/global features only | `Delta P@50_peer > 0` | The purpose of peer normalization is to improve prioritization beyond raw values. The ablation tests exactly that claim. |
| **Future-state classification** | **Precision@50 of the risk score** | Rank pages only by predicted future-state risk and compare at the same K with the frozen simple risk/rule baseline | `P@50_risk model > P@50_risk baseline` | The classifier is used as a ranking score, not merely to produce hard labels. Precision@50 measures whether its highest-risk pages are actually enriched for future events. |
| **Future-change magnitude estimation** | **MAE Skill Score** | `1 - MAE_model / MAE_training-median-baseline` on held-out future magnitude | **MAE Skill Score > 0** | This asks whether the magnitude model improves on the correct trivial constant predictor for absolute-error loss. It does **not** assume a linear model or linear relationship. |
| **Impact estimation** | **Delta NDCG@50** | NDCG@50 of the frozen impact score versus NDCG@50 of the risk-only ranking, using frozen observed future adverse-impact magnitude as graded relevance | `Delta NDCG@50_impact > 0` | Impact estimation exists to move more severe future cases toward the top, not merely more binary positives. NDCG is top-weighted and uses graded severity without assuming a linear predictor. |
| **Final ranking / scoring** | **Precision@50** | Learned final queue versus frozen transparent rule baseline on identical rows and future labels | `Delta P@50_final > 0` | This is the actual operational decision the project is trying to improve. |
| **Reason-code generation** | **Traceability rate** | Recommendations whose reason codes can be exactly regenerated from stored pre-decision inputs divided by all recommendations | **Traceability rate = 100%** | A reason code is useful only if every explanation can be audited back to the evidence that produced it. |
| **Suggested-action generation** | **Evidence-supported action rate** | Suggested actions whose pre-declared evidence requirements are satisfied divided by all suggested actions | **Evidence-supported action rate = 100%** | The current data contain no causal intervention labels. The strongest honest requirement now is therefore that every suggested action is logically supported by its measured evidence. |

### Secondary diagnostics — useful, but never substitutes for the primary metric

| Sub-task | Secondary diagnostics | Faults they help expose |
|---|---|---|
| Signal analysis | sample size, missingness by group, univariate Average Precision lift, direction across time/client slices | sparse cells, missingness artifacts, unstable signals, one-slice effects |
| Clustering | silhouette score, cluster-size balance, bootstrap/refit stability, archetype profiles | geometrically weak clusters, tiny clusters, initialization sensitivity, uninterpretable groups |
| Peer-relative abnormality | outcome rate by abnormality decile, Lift@10%, raw-vs-peer plots | one-bin accidents, bad peer definitions, extreme-value artifacts |
| Classification | Average Precision, Recall@50, reliability/calibration plot, errors by client/time slice | poor global discrimination, low coverage, miscalibrated probabilities, subgroup failure |
| Magnitude | median absolute error, residual/error distribution, errors by client/volume slice | tail failures, systematic over/under-prediction, subgroup instability |
| Impact | captured realized impact in top 50, Precision@50, rank plots | severity score dominated by a few extreme rows or improving severity while destroying event precision |
| Final ranking | Precision@20, Precision@100, Recall@50, NDCG@50, base rate, per-client/time results | cherry-picked K, poor coverage, severity misses, performance concentrated in a few clients/periods |
| Reason codes | reason-code coverage/frequency and manual spot checks | one generic reason dominating, missing explanations, incorrect mappings |
| Suggested actions | action distribution, contradiction checks, reviewer appropriateness and inter-rater agreement **only if real reviewer labels later exist** | degenerate action mapping, evidence/action contradictions, poor human validity |

### Why these metrics avoid the earlier problem

The primary metrics above do **not** require the underlying relationship between features and outcomes to be linear. They also avoid making an intrinsic statistic such as correlation, silhouette or calibration error the reason an upstream stage survives.

For **signal analysis, clustering and peer-relative analysis**, the decisive test is a **paired ablation against the final queue metric**: keep the downstream method fixed, remove only that stage, and see whether Precision@50 gets worse. This makes those stages accountable to the final decision rather than to a convenient internal statistic.

For **classification**, Precision@50 directly matches the queue capacity and does not require choosing a probability cutoff.

For **magnitude estimation**, MAE Skill Score evaluates absolute prediction error relative to a training-only constant baseline. MAE specifies the error cost; it does not assume that the predictor or relationship is linear.

For **impact estimation**, NDCG@50 is used because this stage specifically claims to rank **severity**, not just binary event occurrence. The graded relevance definition must be frozen before testing.

### Fault-finding rules

1. **One primary metric is declared before results for every stage. It cannot be swapped after seeing performance.**
2. **Ablations change one stage only.** Same rows, same target, same downstream learner, same split and same K.
3. **Upstream stages are removable.** If signal analysis, clustering or peer-relative context does not improve its primary ablation metric, it does not get kept merely because a secondary diagnostic looks attractive.
4. **All feature construction occurs inside the training side of the split.** No clustering, peer baselines, feature selection, threshold choice or scaling may learn from the evaluation outcome window.
5. **The rule baseline is frozen before learned-model comparison.**
6. **Primary metrics are calculated on future outcomes unavailable at the decision point.**
7. **Secondary diagnostics cannot rescue a failed primary metric.**
8. **Robustness is reported across client/time slices.** A positive aggregate metric that comes from one dominant client or period is flagged rather than celebrated.
9. **No human-action accuracy is invented.** Reviewer appropriateness is reported only if genuine reviewer labels are collected; causal action efficacy requires a later intervention study.

### Definitions deliberately deferred to Assignment 4

Assignment 4 must freeze, before modeling:

- the exact future adverse-outcome definition;
- the future magnitude definition and sign convention;
- the observed graded future-impact definition used by NDCG;
- eligibility/minimum-volume rules;
- feature and outcome windows;
- and the operational K if a real review capacity becomes available.

Until those are frozen, Section 3 defines **how success will be judged**, not fabricated success results.

In [ ]:
# One-primary-metric-per-subtask contract.
import numpy as np
import pandas as pd

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true, dtype=float)
    scores = np.asarray(scores, dtype=float)
    if len(y_true) != len(scores):
        raise ValueError("y_true and scores must have the same length")
    if not 1 <= k <= len(y_true):
        raise ValueError("k must be between 1 and the number of rows")
    order = np.argsort(-scores, kind="stable")
    return float(y_true[order[:k]].mean())

def delta_precision_at_k(y_true, scores_with_stage, scores_without_stage, k):
    return precision_at_k(y_true, scores_with_stage, k) - precision_at_k(y_true, scores_without_stage, k)

def mae_skill_score(y_true, predictions, training_median):
    y_true = np.asarray(y_true, dtype=float)
    predictions = np.asarray(predictions, dtype=float)
    model_mae = float(np.mean(np.abs(y_true - predictions)))
    baseline_mae = float(np.mean(np.abs(y_true - float(training_median))))
    return np.nan if baseline_mae == 0 else 1.0 - model_mae / baseline_mae

def ndcg_at_k(relevance, scores, k):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    if len(relevance) != len(scores):
        raise ValueError("relevance and scores must have the same length")
    if not 1 <= k <= len(relevance):
        raise ValueError("k must be between 1 and the number of rows")
    ranked = relevance[np.argsort(-scores, kind="stable")[:k]]
    ideal = np.sort(relevance)[::-1][:k]
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = float(np.sum(ranked * discounts))
    idcg = float(np.sum(ideal * discounts))
    return np.nan if idcg == 0 else dcg / idcg

def traceability_rate(is_traceable):
    x = np.asarray(is_traceable, dtype=bool)
    return np.nan if len(x) == 0 else float(x.mean())

def evidence_supported_action_rate(is_supported):
    x = np.asarray(is_supported, dtype=bool)
    return np.nan if len(x) == 0 else float(x.mean())

metric_contract = pd.DataFrame([
    ("signal analysis", "Delta Precision@50 from validated signal set", "> 0"),
    ("clustering / archetypes", "Delta Precision@50 from cluster-derived context", "> 0"),
    ("peer-relative abnormality", "Delta Precision@50 from peer-relative features", "> 0"),
    ("future-state classification", "Precision@50 of risk score", "beats frozen simple risk/rule baseline"),
    ("future-change magnitude", "MAE Skill Score", "> 0"),
    ("impact estimation", "Delta NDCG@50", "> 0 versus risk-only ranking"),
    ("final ranking / scoring", "Precision@50", "beats frozen transparent rule baseline"),
    ("reason-code generation", "Traceability rate", "= 100%"),
    ("suggested-action generation", "Evidence-supported action rate", "= 100%"),
], columns=["sub_task", "primary_metric", "success_rule"])

primary_k = 50
starter_proxy_rate = float(proxy_frame["decline_proxy"].mean())

display(metric_contract)
print(f"Project primary metric: Precision@{primary_k}")
print("Project success: learned final queue beats the frozen rule baseline at the same K.")
print(f"Starter proxy prevalence (plumbing only): {starter_proxy_rate:.1%}")
print("Secondary metrics diagnose faults; they never replace a failed primary metric.")
print("Assignment 4 must freeze the future target, magnitude, realized impact, windows and eligibility rules.")


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.